# Dataset Code Creation - Race

In [ ]:
import fastf1 as ff1
import pandas as pd
from pathlib import Path

In [ ]:
def current_team_name(old_team_name):
    match old_team_name:
        case 'Toro Rosso' | 'AlphaTauri' | 'RB':
            return 'Racing Bulls'
        case 'Sauber' | 'Alfa Romeo Racing' | 'Alfa Romeo':
            return 'Kick Sauber'
        case 'Renault':
            return 'Alpine'
        case 'Racing Point':
            return 'Aston Martin'
        case _:
            return old_team_name

In [ ]:
def race_info(season, file_name=None):

    schedule = ff1.get_event_schedule(season)
    num_rounds = schedule['RoundNumber'].max()
    
    race_times = []
    
    for rnd in range (1, num_rounds + 1):
        race = ff1.get_session(season, rnd, 'R')
        race.load()
        laps = race.laps
        results = race.results
        drivers = [race.get_driver(drv_num)['Abbreviation'] for drv_num in race.drivers] 
        num_laps = int(laps['LapNumber'].max()) 
    
        for drv in drivers:
            drv_laps = race.laps.pick_drivers(drv)

            valid_laps = drv_laps.dropna(subset=['LapTime'])

            if len(valid_laps) == 0:
                print(f"Driver: {drv}, Grand Prix: {race.event['EventName']} no valid race laps")
                continue

            drv_total_time = valid_laps['LapTime'].sum()
            completed_laps = int(valid_laps["LapNumber"].max())
            perc_race_comp = completed_laps/num_laps
            if perc_race_comp < 0.75:
                total_race_time = None
            else:
                mean_lap_time = drv_total_time / completed_laps
                total_race_time = (mean_lap_time * num_laps).total_seconds() 
    
            result = results[results['Abbreviation'] == drv].iloc[0]

            race_times.append({
                'Season' : season,
                'Round' : rnd,
                'Season_Round' : f"{season}-{rnd:02d}",
                'Grand_Prix' : race.event['EventName'],
                'Driver' : drv,
                'Team' : current_team_name(valid_laps['Team'].iloc[0]),
                'Race_Position' : int(result['Position']),
                'Status' : result['Status'], 
                'Completed_Laps' : completed_laps,
                'Max_Laps' : num_laps, 
                'Total_Race_Time' : total_race_time})
    
    df_race_info = pd.DataFrame(race_times)
    
    df_race_info['Pole_Race_Time'] = df_race_info.groupby('Round')['Total_Race_Time'].transform('min') 
    df_race_info['Normalised_Race_Pole_Gap'] = ((df_race_info['Total_Race_Time'] / df_race_info['Pole_Race_Time']) - 1)

    df_race_info = df_race_info[[
        'Season',
        'Round',
        'Season_Round',
        'Grand_Prix',
        'Driver',
        'Team',
        'Status',
        'Completed_Laps',
        'Max_Laps',
        'Race_Position',
        'Pole_Race_Time',
        'Total_Race_Time',
        'Normalised_Race_Pole_Gap']]
    
    if file_name is None:
        return df_race_info
    
    file = Path(f"{file_name}.csv")

    if file.exists():
        df_exist = pd.read_csv(file)
        df_add = pd.concat([df_exist, df_race_info], ignore_index=True)
        df_add = df_add.drop_duplicates(subset=['Season_Round', 'Driver'], keep='last')
        df_add.to_csv(file, index=False)
    else:
        df_race_info.to_csv(file, index=False)
    
    return df_race_info

In [ ]:
race_dataset = race_info(season = 2021, file_name = 'X_season_race_dataset')